# PlantDoc AI — Kaggle Training Notebook

**Mục tiêu:** Huấn luyện model phân loại bệnh lá cây trên GPU Kaggle.  
**Workflow:** Mỗi lần chạy notebook = 1 experiment hoàn chỉnh → tạo ra gói artifact có thể tải về, so sánh, và chọn best model đưa về nhánh `main`.

**Thiết kế chống mất dữ liệu:**
- `last.pt` được lưu **sau mỗi epoch** → runtime ngắt bất kỳ lúc nào cũng giữ được tiến trình.
- `best.pt` được lưu khi metric chính cải thiện.
- Hỗ trợ **resume training** từ `last.pt` khi khởi động lại notebook.
- Artifact cuối cùng được zip để dễ tải về.

---

## Mục lục
1. Cấu hình Experiment
2. Setup môi trường Kaggle
3. Chuẩn bị dữ liệu & đường dẫn
4. Dataset & DataLoader
5. Khởi tạo Model
6. Optimizer / Scheduler / Loss
7. Training & Validation Loop
8. Đánh giá & Trực quan hóa
9. Export Artifact

---
## 1. Cấu hình Experiment

**Đây là cell duy nhất bạn cần thay đổi giữa các lần train.**  
Tất cả tham số quan trọng đều tập trung tại đây.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║              CẤU HÌNH EXPERIMENT - CHỈ SỬA Ở ĐÂY           ║
# ╚══════════════════════════════════════════════════════════════╝

CONFIG = {
    # ── Tên thí nghiệm ──────────────────────────────────────────
    # Đổi tên mỗi lần train để artifact không bị ghi đè
    "experimentName": "mobilenetv2_baseline_v1",
    "notes": "Baseline đầu tiên, freeze backbone, 15 epoch",

    # ── Model ────────────────────────────────────────────────────
    # Tên backbone theo thư viện timm: mobilenetv2_100, efficientnet_b0, resnet50, ...
    "modelName": "mobilenetv2_100",
    "inputSize": 224,
    "freezeBackbone": True,
    "unfreezeFromEpoch": None,  # Đặt số epoch (VD: 6) để tự động unfreeze backbone

    # ── Training ─────────────────────────────────────────────────
    "batchSize": 32,
    "numEpochs": 15,
    "learningRate": 1e-3,
    "weightDecay": 1e-4,
    "optimizerName": "Adam",       # Adam | AdamW | SGD
    "schedulerName": "StepLR",     # StepLR | CosineAnnealingLR | None
    "schedulerStepSize": 5,
    "schedulerGamma": 0.1,
    "useMixedPrecision": True,     # Bật AMP trên GPU Kaggle
    "gradientClipNorm": 1.0,       # None để tắt

    # ── Dữ liệu ─────────────────────────────────────────────────
    # Kaggle dataset path — sửa theo tên dataset bạn đã Add
    "kaggleDatasetSlug": "plantvillage-dataset",  # Tên thư mục trong /kaggle/input/
    "trainSubDir": "train",       # Thư mục con chứa ảnh train (bên trong dataset)
    "valSubDir": "val",           # Thư mục con chứa ảnh val
    "numWorkers": 2,
    "seed": 42,

    # ── Metrics & Checkpointing ──────────────────────────────────
    "topK": 3,
    "metricToSelectBest": "val_f1_macro",  # val_f1_macro | val_acc | val_loss
    "earlyStoppingPatience": None,         # Số epoch chờ, None để tắt

    # ── Resume ───────────────────────────────────────────────────
    "resumeFromCheckpoint": None,  # Đặt path tới last.pt nếu muốn resume
}

print("[OK] Cấu hình experiment đã sẵn sàng.")
print(f"     Experiment: {CONFIG['experimentName']}")
print(f"     Model:      {CONFIG['modelName']}")
print(f"     Epochs:     {CONFIG['numEpochs']}")
print(f"     Backbone:   {'Freeze' if CONFIG['freezeBackbone'] else 'Unfreeze'}")

---
## 2. Setup môi trường Kaggle

In [ ]:
import os
import sys
import json
import csv
import time
import random
import shutil
import warnings
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")  # Backend không cần display

# Cài thêm thư viện nếu chưa có
try:
    import timm
except ImportError:
    os.system("pip install -q timm")
    import timm

try:
    from sklearn.metrics import classification_report, confusion_matrix, f1_score
except ImportError:
    os.system("pip install -q scikit-learn")
    from sklearn.metrics import classification_report, confusion_matrix, f1_score

try:
    import seaborn as sns
except ImportError:
    os.system("pip install -q seaborn")
    import seaborn as sns

try:
    from tqdm.auto import tqdm
except ImportError:
    os.system("pip install -q tqdm")
    from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

print(f"PyTorch: {torch.__version__}")
print(f"timm:    {timm.__version__}")
print(f"Python:  {sys.version.split()[0]}")

In [ ]:
# ── Seed & Device ────────────────────────────────────────────────────────────

def set_seed(seed: int):
    """Đặt seed toàn cục để kết quả tái lập được."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 3. Chuẩn bị đường dẫn dữ liệu & cấu trúc artifact

In [ ]:
# ── Đường dẫn dữ liệu ────────────────────────────────────────────────────────
# Kaggle mount dataset tại /kaggle/input/<tên-dataset>/
# Cấu trúc kỳ vọng:
#   /kaggle/input/<slug>/<trainSubDir>/Apple___Apple_scab/*.jpg
#   /kaggle/input/<slug>/<valSubDir>/Apple___Apple_scab/*.jpg

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

IS_KAGGLE = KAGGLE_INPUT.exists()

if IS_KAGGLE:
    DATA_ROOT = KAGGLE_INPUT / CONFIG["kaggleDatasetSlug"]
    # Liệt kê nội dung dataset để debug
    print(f"[Kaggle] Dataset root: {DATA_ROOT}")
    if DATA_ROOT.exists():
        print(f"         Nội dung: {[x.name for x in sorted(DATA_ROOT.iterdir())[:15]]}")
    else:
        # Thử tìm dataset tự động
        available = [x.name for x in KAGGLE_INPUT.iterdir() if x.is_dir()]
        print(f"[CẢNH BÁO] Không tìm thấy '{CONFIG['kaggleDatasetSlug']}'")
        print(f"            Dataset có sẵn: {available}")
        print("            → Hãy sửa CONFIG['kaggleDatasetSlug'] cho đúng!")
else:
    # Chạy trên local — đường dẫn tương đối
    DATA_ROOT = Path("data/extended")
    print(f"[Local] Data root: {DATA_ROOT}")

TRAIN_DIR = DATA_ROOT / CONFIG["trainSubDir"]
VAL_DIR = DATA_ROOT / CONFIG["valSubDir"]

print(f"Train dir: {TRAIN_DIR}  (tồn tại: {TRAIN_DIR.exists()})")
print(f"Val dir:   {VAL_DIR}  (tồn tại: {VAL_DIR.exists()})")

In [ ]:
# ── Cấu trúc thư mục artifact ────────────────────────────────────────────────
# Mỗi experiment tạo ra một gói hoàn chỉnh, dễ dàng so sánh và tải về.

OUTPUT_ROOT = KAGGLE_WORKING if IS_KAGGLE else Path("artifacts")
EXP_DIR = OUTPUT_ROOT / CONFIG["experimentName"]
CKPT_DIR = EXP_DIR / "checkpoints"
LOG_DIR = EXP_DIR / "logs"
FIG_DIR = EXP_DIR / "figures"

for d in [CKPT_DIR, LOG_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Artifact root: {EXP_DIR}")
print(f"  checkpoints/ : {CKPT_DIR}")
print(f"  logs/         : {LOG_DIR}")
print(f"  figures/      : {FIG_DIR}")

---
## 4. Dataset & DataLoader

In [ ]:
# ── Hằng số chuẩn hóa ImageNet ───────────────────────────────────────────────
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def build_transforms(input_size: int, is_train: bool):
    """
    Tạo transform cho train hoặc eval.
    
    Augmentation thiết kế theo nguyên tắc disease-aware:
    - Hue bị KHÓA ở 0: màu sắc bệnh là tín hiệu chẩn đoán cốt lõi
    - Rotation 180°: hướng lá ngẫu nhiên trong ảnh thực địa
    - Crop scale >= 0.7: không crop mất tổn thương trên lá nhỏ
    - Blur nhẹ: mô phỏng rung tay / ống kính ướt
    - Erasing vùng nhỏ (2-8%): chống memorise background, không xóa tổn thương
    """
    if is_train:
        return transforms.Compose([
            transforms.RandomResizedCrop(input_size, scale=(0.70, 1.0), ratio=(0.85, 1.18)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=180),
            transforms.ColorJitter(
                brightness=0.25,
                contrast=0.20,
                saturation=0.10,
                hue=0.0,          # KHÓA — không đổi màu bệnh
            ),
            transforms.RandomApply(
                [transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.0))],
                p=0.20,
            ),
            transforms.RandomApply(
                [transforms.RandomAdjustSharpness(sharpness_factor=1.5)],
                p=0.20,
            ),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            transforms.RandomErasing(
                p=0.20, scale=(0.02, 0.08), ratio=(1.0, 1.0), value=0,
            ),
        ])
    else:
        return transforms.Compose([
            transforms.Resize(int(input_size * 1.14)),
            transforms.CenterCrop(input_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])


class PlantDiseaseDataset(Dataset):
    """
    Dataset cho phân loại bệnh lá cây.
    Đọc từ cấu trúc thư mục: root/class_name/*.jpg
    """
    def __init__(self, root_dir, transform=None, class_to_idx=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        
        # Quét class folders
        if class_to_idx is not None:
            self.class_to_idx = class_to_idx
        else:
            class_names = sorted([
                d.name for d in self.root_dir.iterdir()
                if d.is_dir() and not d.name.startswith(".")
            ])
            self.class_to_idx = {name: i for i, name in enumerate(class_names)}
        
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}
        
        # Thu thập tất cả mẫu
        self.samples = []
        for class_name, label_id in self.class_to_idx.items():
            class_dir = self.root_dir / class_name
            if not class_dir.exists():
                continue
            for img_path in class_dir.rglob("*"):
                if img_path.suffix.lower() in IMG_EXTS:
                    self.samples.append((str(img_path), label_id))
        
        if len(self.samples) == 0:
            raise ValueError(f"Không tìm thấy ảnh nào trong: {root_dir}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Không load được ảnh: {img_path}. Lỗi: {e}")
        if self.transform:
            img = self.transform(img)
        return img, label


def build_dataloaders(config):
    """Tạo DataLoader cho train và val, trả về class mapping."""
    train_tf = build_transforms(config["inputSize"], is_train=True)
    val_tf = build_transforms(config["inputSize"], is_train=False)
    
    train_ds = PlantDiseaseDataset(TRAIN_DIR, transform=train_tf)
    # Dùng chung class_to_idx từ train để đảm bảo mapping nhất quán
    val_ds = PlantDiseaseDataset(VAL_DIR, transform=val_tf, class_to_idx=train_ds.class_to_idx)
    
    pin = DEVICE.type == "cuda"
    train_loader = DataLoader(
        train_ds, batch_size=config["batchSize"], shuffle=True,
        num_workers=config["numWorkers"], pin_memory=pin, drop_last=False,
    )
    val_loader = DataLoader(
        val_ds, batch_size=config["batchSize"], shuffle=False,
        num_workers=config["numWorkers"], pin_memory=pin, drop_last=False,
    )
    
    return train_loader, val_loader, train_ds.class_to_idx, train_ds.idx_to_class


# ── Khởi tạo ─────────────────────────────────────────────────────────────────
train_loader, val_loader, class_to_idx, idx_to_class = build_dataloaders(CONFIG)
class_names = [idx_to_class[i] for i in range(len(idx_to_class))]
num_classes = len(class_names)

print(f"Số lớp:       {num_classes}")
print(f"Mẫu train:    {len(train_loader.dataset)}")
print(f"Mẫu val:      {len(val_loader.dataset)}")
print(f"Batches/epoch: train={len(train_loader)}, val={len(val_loader)}")

# Phân bố mẫu theo class
train_label_counts = Counter([s[1] for s in train_loader.dataset.samples])
min_class = min(train_label_counts, key=train_label_counts.get)
max_class = max(train_label_counts, key=train_label_counts.get)
print(f"Class ít nhất: {idx_to_class[min_class]} ({train_label_counts[min_class]} mẫu)")
print(f"Class nhiều nhất: {idx_to_class[max_class]} ({train_label_counts[max_class]} mẫu)")

In [ ]:
# ── Lưu class mapping ────────────────────────────────────────────────────────

def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

save_json(EXP_DIR / "class_to_idx.json", class_to_idx)
save_json(EXP_DIR / "idx_to_class.json", {str(k): v for k, v in idx_to_class.items()})
save_json(EXP_DIR / "config.json", CONFIG)

print(f"[OK] Đã lưu class mapping và config vào {EXP_DIR}")

---
## 5. Khởi tạo Model

In [ ]:
def build_model(config, num_classes):
    """
    Xây dựng model sử dụng timm.
    Hỗ trợ freeze/unfreeze backbone cho transfer learning.
    """
    model = timm.create_model(
        config["modelName"],
        pretrained=True,
        num_classes=num_classes,
    )
    
    if config["freezeBackbone"]:
        # Đóng băng toàn bộ backbone
        for param in model.parameters():
            param.requires_grad = False
        # Mở khóa lại classifier head
        classifier = model.get_classifier()
        if classifier is not None:
            for param in classifier.parameters():
                param.requires_grad = True
    
    return model.to(DEVICE)


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def unfreeze_backbone(model):
    """Mở khóa toàn bộ tham số cho fine-tuning sâu."""
    for param in model.parameters():
        param.requires_grad = True
    total, trainable = count_parameters(model)
    print(f"[Unfreeze] Tổng: {total:,} | Trainable: {trainable:,}")


model = build_model(CONFIG, num_classes)
total_params, trainable_params = count_parameters(model)

print(f"Model:     {CONFIG['modelName']}")
print(f"Backbone:  {'Freeze' if CONFIG['freezeBackbone'] else 'Unfreeze'}")
print(f"Params:    {total_params:,} (tổng) | {trainable_params:,} (train được)")

---
## 6. Optimizer / Scheduler / Loss

In [ ]:
# ── Loss ──────────────────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()

# ── Optimizer ─────────────────────────────────────────────────────────────────
trainable_params_list = [p for p in model.parameters() if p.requires_grad]

if CONFIG["optimizerName"] == "Adam":
    optimizer = torch.optim.Adam(
        trainable_params_list, lr=CONFIG["learningRate"], weight_decay=CONFIG["weightDecay"]
    )
elif CONFIG["optimizerName"] == "AdamW":
    optimizer = torch.optim.AdamW(
        trainable_params_list, lr=CONFIG["learningRate"], weight_decay=CONFIG["weightDecay"]
    )
elif CONFIG["optimizerName"] == "SGD":
    optimizer = torch.optim.SGD(
        trainable_params_list, lr=CONFIG["learningRate"],
        momentum=0.9, weight_decay=CONFIG["weightDecay"]
    )
else:
    raise ValueError(f"Optimizer không được hỗ trợ: {CONFIG['optimizerName']}")

# ── Scheduler ─────────────────────────────────────────────────────────────────
scheduler = None
if CONFIG["schedulerName"] == "StepLR":
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=CONFIG["schedulerStepSize"],
        gamma=CONFIG["schedulerGamma"],
    )
elif CONFIG["schedulerName"] == "CosineAnnealingLR":
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CONFIG["numEpochs"]
    )

# ── Mixed Precision ──────────────────────────────────────────────────────────
scaler = torch.amp.GradScaler("cuda") if CONFIG["useMixedPrecision"] and DEVICE.type == "cuda" else None

print(f"Loss:      CrossEntropyLoss")
print(f"Optimizer: {CONFIG['optimizerName']} (lr={CONFIG['learningRate']})")
print(f"Scheduler: {CONFIG['schedulerName'] or 'None'}")
print(f"AMP:       {'Bật' if scaler else 'Tắt'}")

---
## 7. Training & Validation Loop

**Chiến lược chống mất dữ liệu:**
- `last.pt` lưu sau **mỗi epoch** → khôi phục được 100% tiến trình.
- `best.pt` lưu khi metric chính cải thiện.
- `metrics.csv` append mỗi epoch → luôn có lịch sử training.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# HÀM CORE: TRAIN / VAL / CHECKPOINT
# ══════════════════════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, criterion, optimizer, scaler, device, grad_clip=None):
    """Chạy 1 epoch training, trả về dict metrics."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []
    
    pbar = tqdm(loader, desc="  Train", leave=False)
    for images, targets in pbar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        if scaler is not None:
            with torch.amp.autocast("cuda"):
                logits = model(images)
                loss = criterion(logits, targets)
            scaler.scale(loss).backward()
            if grad_clip:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, targets)
            loss.backward()
            if grad_clip:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
        
        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        _, preds = logits.max(1)
        correct += preds.eq(targets).sum().item()
        total += batch_size
        
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
        
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct/total:.4f}")
    
    avg_loss = running_loss / total
    acc = correct / total
    f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    
    return {"loss": avg_loss, "acc": acc, "f1_macro": f1}


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    """Chạy 1 epoch validation, trả về dict metrics + predictions."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []
    
    pbar = tqdm(loader, desc="  Val  ", leave=False)
    for images, targets in pbar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        
        logits = model(images)
        loss = criterion(logits, targets)
        
        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        _, preds = logits.max(1)
        correct += preds.eq(targets).sum().item()
        total += batch_size
        
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
    
    avg_loss = running_loss / total
    acc = correct / total
    f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    
    return {"loss": avg_loss, "acc": acc, "f1_macro": f1, "preds": all_preds, "targets": all_targets}


def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, best_metric, config):
    """Lưu checkpoint đầy đủ để resume training."""
    ckpt = {
        "epoch": epoch,
        "best_metric": best_metric,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
        "scaler_state_dict": scaler.state_dict() if scaler else None,
        "config": config,
    }
    torch.save(ckpt, path)


def load_checkpoint(path, model, optimizer=None, scheduler=None, scaler=None):
    """Load checkpoint và trả về epoch + best_metric."""
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer and ckpt.get("optimizer_state_dict"):
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler and ckpt.get("scheduler_state_dict"):
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if scaler and ckpt.get("scaler_state_dict"):
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    return ckpt.get("epoch", 0), ckpt.get("best_metric", float("-inf"))

print("[OK] Các hàm core đã sẵn sàng.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# VÒNG LẶP TRAINING CHÍNH
# ══════════════════════════════════════════════════════════════════════════════

# Ánh xạ metric name → key trong dict kết quả
METRIC_MAP = {
    "val_f1_macro": ("val", "f1_macro", "higher"),
    "val_acc":      ("val", "acc",      "higher"),
    "val_loss":     ("val", "loss",     "lower"),
}

metric_cfg = METRIC_MAP[CONFIG["metricToSelectBest"]]
best_metric = float("-inf") if metric_cfg[2] == "higher" else float("inf")
best_epoch = 0
start_epoch = 1
no_improve_count = 0  # Đếm cho early stopping

# ── Resume training nếu có ───────────────────────────────────────────────────
if CONFIG["resumeFromCheckpoint"]:
    resume_path = Path(CONFIG["resumeFromCheckpoint"])
    if resume_path.exists():
        print(f"[Resume] Đang load checkpoint: {resume_path}")
        loaded_epoch, loaded_best = load_checkpoint(
            resume_path, model, optimizer, scheduler, scaler
        )
        start_epoch = loaded_epoch + 1
        best_metric = loaded_best
        print(f"[Resume] Tiếp tục từ epoch {start_epoch}, best metric = {loaded_best:.6f}")
    else:
        print(f"[Resume] Không tìm thấy checkpoint: {resume_path}. Bắt đầu từ đầu.")

# ── CSV logger ────────────────────────────────────────────────────────────────
csv_path = LOG_DIR / "metrics.csv"
csv_fields = ["epoch", "train_loss", "train_acc", "train_f1_macro",
              "val_loss", "val_acc", "val_f1_macro", "lr", "epoch_time"]

if start_epoch == 1:  # Chỉ viết header nếu bắt đầu mới
    with open(csv_path, "w", newline="") as f:
        csv.writer(f).writerow(csv_fields)

all_train_metrics = []
all_val_metrics = []
train_start_time = time.time()

print("="*70)
print(f"  BẮT ĐẦU TRAINING: {CONFIG['experimentName']}")
print(f"  Epochs: {start_epoch} → {CONFIG['numEpochs']}")
print(f"  Metric chọn best: {CONFIG['metricToSelectBest']}")
print("="*70)

for epoch in range(start_epoch, CONFIG["numEpochs"] + 1):
    epoch_start = time.time()
    
    # ── Auto-unfreeze backbone ────────────────────────────────────────────
    if CONFIG["unfreezeFromEpoch"] and epoch == CONFIG["unfreezeFromEpoch"]:
        print(f"\n[Epoch {epoch}] AUTO-UNFREEZE backbone!")
        unfreeze_backbone(model)
        # Cập nhật lại optimizer với tất cả tham số
        trainable_params_list = [p for p in model.parameters() if p.requires_grad]
        for pg in optimizer.param_groups:
            pg["lr"] = CONFIG["learningRate"] * 0.1  # LR nhỏ hơn khi fine-tune
        optimizer.param_groups[0]["params"] = trainable_params_list
    
    # ── Train ─────────────────────────────────────────────────────────────
    train_m = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, DEVICE,
        grad_clip=CONFIG["gradientClipNorm"],
    )
    
    # ── Validate ──────────────────────────────────────────────────────────
    val_m = validate_one_epoch(model, val_loader, criterion, DEVICE)
    
    # ── Scheduler step ────────────────────────────────────────────────────
    if scheduler:
        scheduler.step()
    
    current_lr = optimizer.param_groups[0]["lr"]
    epoch_time = time.time() - epoch_start
    
    all_train_metrics.append(train_m)
    all_val_metrics.append(val_m)
    
    # ── Log to CSV ────────────────────────────────────────────────────────
    with open(csv_path, "a", newline="") as f:
        csv.writer(f).writerow([
            epoch,
            f"{train_m['loss']:.6f}", f"{train_m['acc']:.6f}", f"{train_m['f1_macro']:.6f}",
            f"{val_m['loss']:.6f}", f"{val_m['acc']:.6f}", f"{val_m['f1_macro']:.6f}",
            f"{current_lr:.8f}", f"{epoch_time:.1f}",
        ])
    
    # ── Print ─────────────────────────────────────────────────────────────
    print(
        f"Epoch {epoch:>3}/{CONFIG['numEpochs']}  "
        f"train[loss={train_m['loss']:.4f} acc={train_m['acc']:.4f} f1={train_m['f1_macro']:.4f}]  "
        f"val[loss={val_m['loss']:.4f} acc={val_m['acc']:.4f} f1={val_m['f1_macro']:.4f}]  "
        f"lr={current_lr:.6f}  {epoch_time:.0f}s"
    )
    
    # ── Xác định metric hiện tại để so sánh ──────────────────────────────
    current_val_metric = val_m[metric_cfg[1]]
    is_better = (current_val_metric > best_metric) if metric_cfg[2] == "higher" else (current_val_metric < best_metric)
    
    # ── Lưu last.pt SAU MỖI EPOCH — phòng runtime ngắt ──────────────────
    save_checkpoint(
        CKPT_DIR / "last.pt", model, optimizer, scheduler, scaler,
        epoch, best_metric, CONFIG,
    )
    
    # ── Lưu best.pt khi metric cải thiện ─────────────────────────────────
    if is_better:
        best_metric = current_val_metric
        best_epoch = epoch
        no_improve_count = 0
        save_checkpoint(
            CKPT_DIR / "best.pt", model, optimizer, scheduler, scaler,
            epoch, best_metric, CONFIG,
        )
        print(f"  ★ Best model mới! {CONFIG['metricToSelectBest']} = {best_metric:.6f}")
    else:
        no_improve_count += 1
    
    # ── Early stopping ────────────────────────────────────────────────────
    if CONFIG["earlyStoppingPatience"] and no_improve_count >= CONFIG["earlyStoppingPatience"]:
        print(f"\n[Early Stopping] Không cải thiện sau {CONFIG['earlyStoppingPatience']} epoch. Dừng training.")
        break

total_train_time = time.time() - train_start_time
print("="*70)
print(f"  TRAINING HOÀN TẤT")
print(f"  Tổng thời gian: {total_train_time/60:.1f} phút")
print(f"  Best epoch:     {best_epoch}")
print(f"  Best {CONFIG['metricToSelectBest']}: {best_metric:.6f}")
print(f"  Checkpoint:     {CKPT_DIR / 'best.pt'}")
print("="*70)

---
## 8. Đánh giá & Trực quan hóa

Load lại best model để đánh giá chính thức trên tập validation.

In [ ]:
# ── Load best model để đánh giá ──────────────────────────────────────────────

best_ckpt_path = CKPT_DIR / "best.pt"
if best_ckpt_path.exists():
    load_checkpoint(best_ckpt_path, model)
    print(f"[OK] Đã load best model từ epoch {best_epoch}")
else:
    print("[CẢNH BÁO] Không tìm thấy best.pt, dùng model hiện tại.")

# ── Chạy validation cuối cùng ────────────────────────────────────────────────
final_val = validate_one_epoch(model, val_loader, criterion, DEVICE)

print(f"\nKết quả best model trên tập val:")
print(f"  Loss:     {final_val['loss']:.4f}")
print(f"  Accuracy: {final_val['acc']:.4f}")
print(f"  F1 Macro: {final_val['f1_macro']:.4f}")

In [ ]:
# ── Classification Report ────────────────────────────────────────────────────

report_text = classification_report(
    final_val["targets"], final_val["preds"],
    target_names=class_names, digits=4,
)
print(report_text)

# Lưu ra file
with open(EXP_DIR / "classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report_text)
print(f"[OK] Đã lưu classification report.")

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────

cm = confusion_matrix(final_val["targets"], final_val["preds"])

fig, ax = plt.subplots(1, 1, figsize=(max(14, num_classes * 0.5), max(12, num_classes * 0.4)))
sns.heatmap(cm, annot=(num_classes <= 30), fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_ylabel("True Label")
ax.set_xlabel("Predicted Label")
ax.set_title(f"Confusion Matrix — {CONFIG['experimentName']}")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()

cm_path = FIG_DIR / "confusion_matrix.png"
fig.savefig(cm_path, dpi=200)
plt.show()
print(f"[OK] Đã lưu confusion matrix → {cm_path}")

In [ ]:
# ── Biểu đồ Loss & Accuracy & F1 theo epoch ─────────────────────────────────

def plot_training_curves(train_metrics, val_metrics, fig_dir, experiment_name):
    """Vẽ loss, accuracy, f1 curves và lưu ra file."""
    epochs = range(1, len(train_metrics) + 1)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Loss
    axes[0].plot(epochs, [m["loss"] for m in train_metrics], "o-", label="Train", markersize=3)
    axes[0].plot(epochs, [m["loss"] for m in val_metrics], "s-", label="Val", markersize=3)
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(epochs, [m["acc"] for m in train_metrics], "o-", label="Train", markersize=3)
    axes[1].plot(epochs, [m["acc"] for m in val_metrics], "s-", label="Val", markersize=3)
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # F1 Macro
    axes[2].plot(epochs, [m["f1_macro"] for m in train_metrics], "o-", label="Train", markersize=3)
    axes[2].plot(epochs, [m["f1_macro"] for m in val_metrics], "s-", label="Val", markersize=3)
    axes[2].set_title("F1 Macro")
    axes[2].set_xlabel("Epoch")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    fig.suptitle(experiment_name, fontsize=14, fontweight="bold")
    plt.tight_layout()
    
    path = fig_dir / "training_curves.png"
    fig.savefig(path, dpi=150)
    plt.show()
    print(f"[OK] Đã lưu training curves → {path}")


plot_training_curves(all_train_metrics, all_val_metrics, FIG_DIR, CONFIG["experimentName"])

In [ ]:
# ── Lưu summary.json — metadata experiment để so sánh nhiều lần train ────────

summary = {
    "experimentName": CONFIG["experimentName"],
    "notes": CONFIG.get("notes", ""),
    "timestamp": datetime.now().isoformat(),
    
    # Model
    "modelName": CONFIG["modelName"],
    "inputSize": CONFIG["inputSize"],
    "totalParams": total_params,
    "trainableParams": trainable_params,
    "freezeBackbone": CONFIG["freezeBackbone"],
    "unfreezeFromEpoch": CONFIG["unfreezeFromEpoch"],
    
    # Training
    "optimizer": CONFIG["optimizerName"],
    "scheduler": CONFIG["schedulerName"],
    "learningRate": CONFIG["learningRate"],
    "batchSize": CONFIG["batchSize"],
    "numEpochsTrained": epoch,  # epoch cuối cùng thực sự chạy
    "numEpochsConfig": CONFIG["numEpochs"],
    "useMixedPrecision": CONFIG["useMixedPrecision"],
    "seed": CONFIG["seed"],
    
    # Kết quả
    "metricToSelectBest": CONFIG["metricToSelectBest"],
    "bestEpoch": best_epoch,
    "bestMetricValue": float(best_metric),
    "finalValLoss": float(final_val["loss"]),
    "finalValAcc": float(final_val["acc"]),
    "finalValF1Macro": float(final_val["f1_macro"]),
    
    # Thời gian
    "totalTrainTimeMinutes": round(total_train_time / 60, 1),
    
    # Dữ liệu
    "numClasses": num_classes,
    "trainSamples": len(train_loader.dataset),
    "valSamples": len(val_loader.dataset),
}

save_json(EXP_DIR / "summary.json", summary)

print("\n" + "="*50)
print("  EXPERIMENT SUMMARY")
print("="*50)
for k, v in summary.items():
    if k not in ("notes", "timestamp"):
        print(f"  {k:<25}: {v}")
print("="*50)

---
## 9. Export Artifact

Zip toàn bộ gói experiment lại để dễ tải về.  
Dù Kaggle runtime ngắt sau cell này, file zip vẫn nằm sẵn trong `/kaggle/working/` → tải qua tab **Output**.

In [ ]:
# ── Liệt kê artifact đã tạo ──────────────────────────────────────────────────

print(f"\nArtifact trong {EXP_DIR}:")
for p in sorted(EXP_DIR.rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / 1e6
        print(f"  {p.relative_to(EXP_DIR)}  ({size_mb:.1f} MB)")

In [ ]:
# ── Zip artifact ──────────────────────────────────────────────────────────────

zip_name = CONFIG["experimentName"]
zip_base = KAGGLE_WORKING if IS_KAGGLE else Path(".")
zip_path = shutil.make_archive(
    str(zip_base / zip_name),  # Tên file (không có .zip)
    "zip",
    root_dir=str(EXP_DIR.parent),
    base_dir=EXP_DIR.name,
)

zip_size = Path(zip_path).stat().st_size / 1e6
print(f"\n[OK] Artifact đã được zip!")
print(f"     File: {zip_path}")
print(f"     Size: {zip_size:.1f} MB")
print(f"")
if IS_KAGGLE:
    print("     → Vào tab 'Output' bên phải Kaggle Notebook để tải file này về.")
else:
    print(f"     → File nằm tại: {zip_path}")

---
## Tiếp theo là gì?

1. **So sánh experiments:** Mở các file `summary.json` từ nhiều lần chạy để so sánh `bestMetricValue`, `finalValF1Macro`.
2. **Chọn best model:** File `best.pt` của experiment có `finalValF1Macro` cao nhất.
3. **Giải nén artifact:** Tải zip về, giải nén, copy `best.pt` + `config.json` + `class_to_idx.json` vào project.
4. **Đưa về `main`:** Chỉ commit code + config vào Git. Checkpoint (`.pt`) lưu riêng (Google Drive, Kaggle Datasets, HuggingFace Hub).

### Các biến thể có thể thử:

| Experiment | Thay đổi CONFIG |
|---|---|
| Baseline freeze | `freezeBackbone=True`, `numEpochs=15` |
| Fine-tune toàn bộ | `freezeBackbone=False`, `learningRate=1e-4` |
| 2-stage | `freezeBackbone=True`, `unfreezeFromEpoch=6`, `numEpochs=20` |
| EfficientNet | `modelName='efficientnet_b0'`, `inputSize=224` |
| EfficientNet lớn hơn | `modelName='efficientnet_b2'`, `inputSize=288` |